In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms 

In [2]:
with sqlite3.connect("../data/pancreas_proteomics.db") as conn:
    df = pd.read_sql("SELECT * FROM imputed_KNN", conn)
    df_ann = pd.read_sql("SELECT [Protein ID], [Annotated Matrisome Category] FROM annotations", conn)
    condition = pd.read_sql("SELECT [Sample ID], Condition FROM expression GROUP BY [Sample ID]", conn)

df_pivot = df.pivot(index='Protein ID', columns='Sample ID', values='Intensity')
df_input = df_pivot.T
#df_input

In [3]:
scaler = StandardScaler()
df_input_scaled = scaler.fit_transform(df_input)
#df_input_scaled.mean()

In [4]:
pca = PCA(n_components=10)
components = pca.fit_transform(df_input_scaled)

In [8]:
#pca.components_.T
loadings_squared = pd.DataFrame(pca.components_[:2,:].T ** 2, columns=['PC1', 'PC2'], index=df_input.columns)
loadings_squared = loadings_squared.reset_index().rename(columns={'index': 'Protein ID'})


In [9]:
merged_loadings = pd.merge(loadings_squared, df_ann, on='Protein ID', how='inner')

In [10]:
variance_contribution = merged_loadings.groupby('Annotated Matrisome Category')[['PC1']].sum() * 100
variance_contribution = variance_contribution.sort_values(by='PC1', ascending=False)

In [11]:
variance_contribution

,PC1
Annotated Matrisome Category,
Non-matrisome,95.130294
ECM Regulators,1.858135
ECM Glycoproteins,1.453777
ECM-affiliated Proteins,0.765786
Secreted Factors,0.420054
Collagens,0.260628
Proteoglycans,0.111328
